## Transformer Model for translation English -> Finnish.

In [134]:
import keras
import tensorflow as tf
import numpy as np
from keras import layers
from keras import ops

In [135]:
text_file = "fin-eng/fin.txt"

with open(text_file, encoding='utf-8') as f:
    lines = f.read().split("\n")[:-1]
text_pairs = []
for line in lines:
    english, finnish, rest = line.split("\t")
    finnish = "[start] " + finnish + " [end]"
    text_pairs.append((english, finnish))

print(text_pairs[:10])

[('Go.', '[start] Mene. [end]'), ('Hi.', '[start] Moro! [end]'), ('Hi.', '[start] Terve. [end]'), ('Run!', '[start] Juokse! [end]'), ('Run!', '[start] Juoskaa! [end]'), ('Run.', '[start] Juokse. [end]'), ('Who?', '[start] Kuka? [end]'), ('Wow!', '[start] Mahtavaa! [end]'), ('Wow!', '[start] Siistiä! [end]'), ('Wow!', '[start] Vau! [end]')]


In [136]:
import random
random.shuffle(text_pairs)
num_val_samples = int(0.15 * len(text_pairs))
num_train_samples = len(text_pairs) - 2 * num_val_samples
train_pairs = text_pairs[:num_train_samples]
val_pairs = text_pairs[num_train_samples:num_train_samples + num_val_samples]
test_pairs = text_pairs[num_train_samples + num_val_samples:]


# import random

# random.seed(42)
# random.shuffle(text_pairs)

# train_ratio = 0.8
# val_ratio = 0.1
# test_ratio = 0.1

# total_size = len(text_pairs)
# train_size = int(total_size * train_ratio)
# val_size = int(total_size * val_ratio)

# train_pairs = text_pairs[:train_size]
# val_pairs = text_pairs[train_size:train_size+val_size]
# test_pairs = text_pairs[train_size+val_size:]

# print(f"Total pairs: {total_size}")
# print(f"Training pairs: {len(train_pairs)}")
# print(f"Validation pairs: {len(val_pairs)}")
# print(f"Testing pairs: {len(test_pairs)}")

In [137]:
import string
import re

strip_chars = string.punctuation
strip_chars = strip_chars.replace("[", "")
strip_chars = strip_chars.replace("]", "")


def custom_standardization(input_string):
    lowercase = tf.strings.lower(input_string)
    return tf.strings.regex_replace(
        lowercase, f"[{re.escape(strip_chars)}]", "")

In [138]:
vocab_size = 15000
sequence_length = 20

source_vectorization = layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length,
)

target_vectorization = layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length + 1,
    # standardize=custom_standardization,
)


train_english_texts = [pair[0] for pair in train_pairs]
train_finnish_texts = [pair[1] for pair in train_pairs]
source_vectorization.adapt(train_english_texts)
target_vectorization.adapt(train_finnish_texts)




In [139]:
batch_size = 64

def format_dataset(eng, fin):
    eng = source_vectorization(eng)
    fin = target_vectorization(fin)
    return ({
        "english": eng,
        "finnish": fin[:, :-1],
    }, fin[:, 1:])

def make_dataset(pairs):
    eng_texts, fin_texts = zip(*pairs)
    eng_texts = list(eng_texts)
    fin_texts = list(fin_texts)
    dataset = tf.data.Dataset.from_tensor_slices((eng_texts, fin_texts))
    dataset = dataset.batch(batch_size)
    dataset = dataset.map(format_dataset,  num_parallel_calls=4)
    return dataset.shuffle(2048).prefetch(16).cache()

train_ds = make_dataset(train_pairs)
val_ds = make_dataset(val_pairs)

for inputs, targets in train_ds.take(1):
    print(f"inputs['english'].shape: {inputs['english'].shape}")
    print(f"inputs['finnish'].shape: {inputs['finnish'].shape}")

inputs['english'].shape: (64, 20)
inputs['finnish'].shape: (64, 20)


2025-04-25 12:16:42.454015: W tensorflow/core/kernels/data/cache_dataset_ops.cc:916] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


In [140]:
class TransformerDecoder(layers.Layer):
    def __init__(self, embed_dim, dense_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.dense_dim = dense_dim
        self.num_heads = num_heads
        self.attention_1 = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.attention_2 = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.dense_proj = keras.Sequential([
            layers.Dense(dense_dim, activation="relu"),
            layers.Dense(embed_dim),]
            )
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()
        self.layernorm_3 = layers.LayerNormalization()
        self.supports_masking = True

    def get_config(self):
        config = super().get_config()
        config.update({
            "embed_dim": self.embed_dim,
            "num_heads": self.num_heads,
            "dense_dim": self.dense_dim,
        })
        return config

    def get_casual_attention_mask(self, inputs):
        input_shape = tf.shape(inputs)
        batch_size, sequence_length = input_shape[0], input_shape[1]
        i = tf.range(sequence_length)[:, tf.newaxis]
        j = tf.range(sequence_length)
        mask = tf.cast(i >= j, dtype="int32")
        mask = tf.reshape(mask, (1, input_shape[1], input_shape[1]))  
        mult = tf.concat([tf.expand_dims(batch_size, -1), tf.constant([1, 1], dtype=tf.int32)], axis=0)
        return tf.tile(mask, mult)
    
    def call(self, inputs, encoder_outputs, mask=None):
        # Create a causal mask for self-attention
        causal_mask = self.get_casual_attention_mask(inputs)

        # Combine with padding mask if provided
        if mask is not None:
            padding_mask = tf.cast(mask[:, tf.newaxis, :], dtype="int32")
            padding_mask = tf.minimum(padding_mask, causal_mask)
        else:
            padding_mask = causal_mask

        # Self-attention (decoder attends to previous tokens)
        attention_output_1 = self.attention_1(
            query=inputs,
            value=inputs,
            key=inputs,
            attention_mask=causal_mask
        )
        attention_output_1 = self.layernorm_1(inputs + attention_output_1)

        # Cross-attention (decoder attends to encoder outputs)
        attention_output_2 = self.attention_2(
            query=attention_output_1,
            value=encoder_outputs,
            key=encoder_outputs,
            attention_mask=padding_mask
        )
        attention_output_2 = self.layernorm_2(attention_output_1 + attention_output_2)

        # Feed-forward network (dense projection)
        proj_output = self.dense_proj(attention_output_2)
        
        # Final layer normalization and return
        return self.layernorm_3(attention_output_2 + proj_output)



In [141]:
class TransformerEncoder(layers.Layer):
    def __init__(self, embed_dim, dense_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.dense_dim = dense_dim
        self.num_heads = num_heads
        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim
        )
        self.dense_proj = keras.Sequential(
            [
                layers.Dense(dense_dim, activation="relu"),
                layers.Dense(embed_dim),
            ]
        )
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()
        self.supports_masking = True

    def call(self, inputs, mask=None):
        if mask is not None:
            padding_mask = ops.cast(mask[:, None, :], dtype="int32")
        else:
            padding_mask = None

        attention_output = self.attention(
            query=inputs, value=inputs, key=inputs, attention_mask=padding_mask
        )
        proj_input = self.layernorm_1(inputs + attention_output)
        proj_output = self.dense_proj(proj_input)
        return self.layernorm_2(proj_input + proj_output)

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "embed_dim": self.embed_dim,
                "dense_dim": self.dense_dim,
                "num_heads": self.num_heads,
            }
        )
        return config

In [142]:
class PositionalEmbedding(layers.Layer):
    def __init__(self, sequence_length, vocab_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.token_embeddings = layers.Embedding(
            input_dim=vocab_size, output_dim=embed_dim
        )
        self.position_embeddings = layers.Embedding(
            input_dim=sequence_length, output_dim=embed_dim
        )
        self.sequence_length = sequence_length
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim

    def call(self, inputs):
        length = ops.shape(inputs)[-1]
        positions = ops.arange(0, length, 1)
        embedded_tokens = self.token_embeddings(inputs)
        embedded_positions = self.position_embeddings(positions)
        return embedded_tokens + embedded_positions

    def compute_mask(self, inputs, mask=None):
        return ops.not_equal(inputs, 0)

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "sequence_length": self.sequence_length,
                "vocab_size": self.vocab_size,
                "embed_dim": self.embed_dim,
            }
        )
        return config

In [143]:
embed_dim = 128
dense_dim = 512
num_heads = 4

encoder_inputs = keras.Input(shape=(None,), dtype="int64", name="english")
x = PositionalEmbedding(sequence_length, vocab_size, embed_dim)(encoder_inputs)
encoder_outputs = TransformerEncoder(embed_dim, dense_dim, num_heads) (x)

decoder_inputs = keras.Input(shape=(None,), dtype="int64", name="finnish")
x = PositionalEmbedding(sequence_length, vocab_size, embed_dim)(decoder_inputs)
x = TransformerDecoder(embed_dim, dense_dim, num_heads) (x, encoder_outputs)
x = layers.Dropout(0.5) (x)

decoder_outputs = layers.Dense(vocab_size, activation="softmax") (x)
transformer = keras.Model([encoder_inputs, decoder_inputs], decoder_outputs)

transformer.compile(
 optimizer="rmsprop",
 loss="sparse_categorical_crossentropy",
 metrics=["accuracy"])
transformer.fit(train_ds, epochs=30, validation_data=val_ds)

Epoch 1/30
791/791 ━━━━━━━━━━━━━━━━━━━━ 82s 101ms/step - accuracy: 0.1207 - loss: 5.4902 - val_accuracy: 0.1570 - val_loss: 3.8075
Epoch 2/30
791/791 ━━━━━━━━━━━━━━━━━━━━ 80s 101ms/step - accuracy: 0.1553 - loss: 3.9814 - val_accuracy: 0.1744 - val_loss: 3.3389
Epoch 3/30
791/791 ━━━━━━━━━━━━━━━━━━━━ 79s 100ms/step - accuracy: 0.1677 - loss: 3.5993 - val_accuracy: 0.1782 - val_loss: 3.1858
Epoch 4/30
791/791 ━━━━━━━━━━━━━━━━━━━━ 79s 100ms/step - accuracy: 0.1755 - loss: 3.3857 - val_accuracy: 0.1825 - val_loss: 3.1077
Epoch 5/30
791/791 ━━━━━━━━━━━━━━━━━━━━ 79s 100ms/step - accuracy: 0.1813 - loss: 3.2469 - val_accuracy: 0.1870 - val_loss: 3.0251
Epoch 6/30
791/791 ━━━━━━━━━━━━━━━━━━━━ 80s 101ms/step - accuracy: 0.1851 - loss: 3.1570 - val_accuracy: 0.1897 - val_loss: 2.9953
Epoch 7/30
791/791 ━━━━━━━━━━━━━━━━━━━━ 79s 100ms/step - accuracy: 0.1893 - loss: 3.0870 - val_accuracy: 0.1921 - val_loss: 2.9505
Epoch 8/30
791/791 ━━━━━━━━━━━━━━━━━━━━ 79s 100ms/step - accuracy: 0.1922 - loss: 3

In [144]:
# transformer.save('english_finnish_transformer.keras')

In [146]:
import numpy as np
fin_vocab = target_vectorization.get_vocabulary()
fin_index_lookup = dict(zip(range(len(fin_vocab)), fin_vocab))
max_decoded_sentence_length = 20

def decode_sequence(input_sequence):
    tokenized_input_sentence = source_vectorization([input_sequence])
    decoded_sentence = "[start]"
    for i in range(max_decoded_sentence_length):
        tokenized_target_sentence = target_vectorization([decoded_sentence])[:, :-1]
        predictions = transformer([tokenized_input_sentence, tokenized_target_sentence])
        sampled_token_index = np.argmax(predictions[0, i, :])
        sampled_token = fin_index_lookup[sampled_token_index]
        decoded_sentence += " " + sampled_token
        if sampled_token == "[end]":
            break
        return decoded_sentence
    
test_eng_texts = [pair[0] for pair in test_pairs]
for _ in range(20):
    input_sentence = random.choice(test_eng_texts)
    print(input_sentence)
    print(decode_sequence(input_sentence))

Can your mother drive a car?
[start] voisitko
You have to stop Tom.
[start] sinun
Is lunch included in this price?
[start] [UNK]
Stand aside.
[start] katso
I found this column interesting.
[start] minusta
I've got a dangerous situation here.
[start] minulla
It's time to talk.
[start] on
It's been a long time since I visited my grandmother.
[start] viime
How's your family?
[start] miten
Tom stayed home.
[start] tom
The searchers could hear faint calls coming from deep within the cave.
[start] [UNK]
We haven't done a thing all week.
[start] emme
How nice!
[start] miten
The president gave up the idea because it was not practical.
[start] [UNK]
They're really cool.
[start] he
This could be a trap.
[start] tämä
What kind of fruit juice do you have?
[start] [UNK]
Tom is working as a waiter, but he's looking for a better job.
[start] tom
Teenagers do a lot of stupid things.
[start] ne
You have quite an imagination.
[start] sinulla
